In [1]:
import sqlite3
import pandas as pd
import yfinance as yf

ipo_clean = pd.read_csv('../data/processed/ipo_clean.csv')
price_history = pd.read_csv('../data/processed/price_history.csv')
nifty50 = pd.read_csv('../data/processed/nifty50_history.csv')

print(ipo_clean.shape)
print(price_history.shape)
print(nifty50.shape)

(397, 11)
(305199, 3)
(2090, 2)


In [2]:
#fix no.1 Premier Energies had wrong NSE symbol, so manual override
ipo_clean.loc[ipo_clean['company_name'] == 'Premier Energies Limited', 'nse_symbol'] = 'PREMIERENE'

ticker_obj = yf.Ticker('PREMIERENE.NS')
hist = ticker_obj.history(start='2024-09-03', end='2026-06-30', auto_adjust=True)
hist = hist.reset_index()
hist['nse_symbol'] = 'PREMIERENE'

hist['Date'] = hist['Date'].astype(str)

price_history = price_history[price_history['nse_symbol'] != 'PREMIER']
price_history = pd.concat([price_history,hist[['nse_symbol','Date','Close']]], ignore_index = True)

In [3]:
#fix 2 Protean eGov's listing_date was BSE's date, not NSE's actual first trading date
protean_first_date = price_history[price_history['nse_symbol'] == 'PROTEAN']['Date'].min()
ipo_clean.loc[ipo_clean['company_name'].str.contains('Protean', na=False), 'listing_date'] = protean_first_date

print(price_history[price_history['nse_symbol'] == 'PREMIERENE'].shape)
print(protean_first_date)

(453, 3)
2025-02-06 00:00:00+05:30


In [4]:
#create a SQLite database
conn = sqlite3.connect('../data/processed/ipo_analysis.db')
ipo_clean.to_sql('ipo_clean',conn,if_exists ='replace',index=False)
price_history.to_sql('price_history',conn,if_exists ='replace',index=False)
nifty50.to_sql('nifty50',conn,if_exists ='replace',index=False)

2090

In [5]:
conn.execute("CREATE INDEX IF NOT EXISTS idx_symbol_date ON price_history(nse_symbol, Date)")
conn.commit()

In [6]:
#calculating percentage of premium
ipo_clean['pop_pct'] = ((ipo_clean['listing_price'] - ipo_clean['issue_price']) / ipo_clean['issue_price']) * 100
ipo_clean.to_sql('ipo_clean',conn,if_exists = 'replace',index = False)
print(ipo_clean['pop_pct'].describe())
ipo_clean[['company_name','listing_date','nse_symbol','pop_pct']].head(10)

count    397.000000
mean      20.646495
std       34.767695
min      -38.888889
25%        0.000000
50%        9.090909
75%       32.027257
max      252.760736
Name: pop_pct, dtype: float64


,company_name,listing_date,nse_symbol,pop_pct
0,Gujarat Kidney & S...,2025-12-30,GKSL,5.921053
1,KSH International ...,2025-12-23,KSHINTL,-3.645833
2,ICICI Prudential A...,2025-12-19,ICICIAMC,20.378753
3,Nephrocare Health ...,2025-12-17,NEPHROPLUS,6.891304
4,Park Medi World Lt...,2025-12-17,PARKHOSPS,-3.950617
5,Wakefit Innovation...,2025-12-15,WAKEFIT,0.000000
6,Corona Remedies Lt...,2025-12-15,CORONA,36.723164
7,Aequs Ltd,2025-12-10,AEQUS,12.903226
8,Meesho Ltd,2025-12-10,MEESHO,45.225225
9,Vidya Wires Ltd,2025-12-10,VIDYAWIRES,0.250000


In [7]:
# -100% pop seems like a source error
print(ipo_clean[ipo_clean['pop_pct'] < -50][['company_name', 'issue_price', 'listing_price', 'pop_pct']])

Empty DataFrame
Columns: [company_name, issue_price, listing_price, pop_pct]
Index: []


In [8]:
#manual overide to correct the listing price
ipo_clean.loc[ipo_clean['company_name'].str.contains('Wakefit'),'listing_price'] = 195
ipo_clean.loc[ipo_clean['company_name'].str.contains('Wework'),'listing_price'] = 650

ipo_clean['pop_pct'] = ((ipo_clean['listing_price'] - ipo_clean['issue_price']) / ipo_clean['issue_price']) * 100
ipo_clean.to_sql('ipo_clean',conn,if_exists = 'replace',index = False)

397

In [9]:
print(ipo_clean['pop_pct'].describe())

count    397.000000
mean      20.646495
std       34.767695
min      -38.888889
25%        0.000000
50%        9.090909
75%       32.027257
max      252.760736
Name: pop_pct, dtype: float64


In [10]:
def get_forward_prices (day_offset,time_period):
    """
    For each company, finds the closing price nearest to (listing_date + day_offset).
    Returns NaN for price and date if the nearest match is more than 9 days away
    (usually meaning the target date hasn't happened yet, relative to our cutoff).
    """
    query = f"""
    WITH target_dates AS (
    SELECT nse_symbol ,listing_date,
       date(listing_date, '+{day_offset} days') AS target_date
    FROM ipo_clean
    ),
    ranked_prices AS (
    SELECT 
        t.nse_symbol,
        t.listing_date,
        t.target_date,
        p.Date AS price_date,
        p.Close AS close_price,
        ABS(julianday(p.Date) - julianday(t.target_date)) AS day_diff,
        ROW_NUMBER() OVER (
            PARTITION BY t.nse_symbol
            ORDER BY ABS(julianday(p.Date) - julianday(t.target_date))ASC
        ) AS rank
        FROM target_dates t
        JOIN price_history p ON t.nse_symbol = p.nse_symbol    
    )
    SELECT 
        nse_symbol, 
        listing_date, 
        target_date, 
        CASE 
            WHEN day_diff >=10 THEN NULL
            ELSE price_date
        END AS price_date,
        CASE 
            WHEN day_diff >=10 THEN NULL
            ELSE close_price
        END AS close_{time_period},
        day_diff
    FROM ranked_prices
    WHERE rank = 1
    """
    return pd.read_sql(query, conn)

In [11]:
periods = [(182, '6mo'), (365, '1yr'), (730, '2yr'), (1095, '3yr')]
results = {}

for day_offset,period in periods:
    print(f"Running {period}-")
    results[period] = get_forward_prices(day_offset,period)
    print(f"Done:{results[period].shape}")

Running 6mo-
Done:(397, 6)
Running 1yr-
Done:(397, 6)
Running 2yr-
Done:(397, 6)
Running 3yr-
Done:(397, 6)


In [12]:
master = ipo_clean[['nse_symbol','company_name','listing_date', 'issue_price', 'listing_price', 'pop_pct']].copy()

for period in ['6mo','1yr','2yr','3yr']:
    price_col = f'close_{period}'
    merge_cols =['nse_symbol',price_col]
    master = master.merge(results[period][merge_cols], on ='nse_symbol', how ='left')

print(master.shape)
master.head(100)

(397, 10)


,nse_symbol,company_name,listing_date,issue_price,listing_price,pop_pct,close_6mo,close_1yr,close_2yr,close_3yr
0,GKSL,Gujarat Kidney & S...,2025-12-30,114,120.75,5.921053,129.330002,NaN,NaN,NaN
1,KSHINTL,KSH International ...,2025-12-23,384,370.00,-3.645833,863.400024,NaN,NaN,NaN
2,ICICIAMC,ICICI Prudential A...,2025-12-19,2165,2606.20,20.378753,3429.000000,NaN,NaN,NaN
3,NEPHROPLUS,Nephrocare Health ...,2025-12-17,460,491.70,6.891304,737.000000,NaN,NaN,NaN
4,PARKHOSPS,Park Medi World Lt...,2025-12-17,162,155.60,-3.950617,254.149994,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
95,AJAXENGG,Ajax Engineering Limited,2025-02-17,629,593.00,-5.723370,698.849976,486.500000,NaN,NaN
96,AGARWALEYE,Dr Agarwal's Healthcare Limited,2025-02-04,402,396.90,-1.268657,459.350006,450.350006,NaN,NaN
97,DENTA,Denta Water & Infrastructure Limited,2025-01-29,294,330.00,12.244898,321.126556,261.850006,NaN,NaN
98,STALLION,Stallion India Fluorochem Limited,2025-01-23,90,120.00,33.333333,113.669998,197.399994,NaN,NaN


In [13]:
#getting %returns
for period in ['6mo','1yr','2yr','3yr']:
    master[f'return_{period}'] = (( master[f'close_{period}']- master['listing_price']) /  master['listing_price'])*100

In [14]:
#getting CAGR(Compound Annual Growth Rate)
years = {'6mo':0.5,'1yr':1,'2yr':2,'3yr':3}

for period,year in years.items():
    master[f'cagr_{period}'] = (( master[f'close_{period}']/ master['listing_price']) ** (1/year) - 1)*100

In [15]:
master.head()

,nse_symbol,company_name,listing_date,issue_price,listing_price,pop_pct,close_6mo,close_1yr,close_2yr,close_3yr,return_6mo,return_1yr,return_2yr,return_3yr,cagr_6mo,cagr_1yr,cagr_2yr,cagr_3yr
0,GKSL,Gujarat Kidney & S...,2025-12-30,114,120.75,5.921053,129.330002,NaN,NaN,NaN,7.105592,NaN,NaN,NaN,14.716077,NaN,NaN,NaN
1,KSHINTL,KSH International ...,2025-12-23,384,370.00,-3.645833,863.400024,NaN,NaN,NaN,133.351358,NaN,NaN,NaN,444.528563,NaN,NaN,NaN
2,ICICIAMC,ICICI Prudential A...,2025-12-19,2165,2606.20,20.378753,3429.000000,NaN,NaN,NaN,31.570869,NaN,NaN,NaN,73.108937,NaN,NaN,NaN
3,NEPHROPLUS,Nephrocare Health ...,2025-12-17,460,491.70,6.891304,737.000000,NaN,NaN,NaN,49.888143,NaN,NaN,NaN,124.664555,NaN,NaN,NaN
4,PARKHOSPS,Park Medi World Lt...,2025-12-17,162,155.60,-3.950617,254.149994,NaN,NaN,NaN,63.335472,NaN,NaN,NaN,166.784763,NaN,NaN,NaN


In [16]:
# Checking for a company old enough to have all 4 periods filled
master[master['close_3yr'].notna()][
    ['company_name', 'return_6mo', 'cagr_6mo', 'return_1yr', 'cagr_1yr', 
     'return_2yr', 'cagr_2yr', 'return_3yr', 'cagr_3yr']
].head(5)

,company_name,return_6mo,cagr_6mo,return_1yr,cagr_1yr,return_2yr,cagr_2yr,return_3yr,cagr_3yr
234,Ideaforge Technologies Limited,-36.357369,-59.496155,-36.276912,-36.276912,-55.432532,-33.241129,-37.862998,-14.667049
235,HMA Agro Industries Limited,-87.921244,-98.541037,-91.407994,-91.407994,-95.061949,-77.778273,-96.435772,-67.090460
236,IKIO Technologies Limited (formerly IKIO Light...,-9.861494,-18.750497,-27.040447,-27.040447,-44.572890,-25.550614,-58.473147,-25.393556
237,Mankind Pharma Limited,37.930251,90.247541,67.833834,67.833834,87.579909,36.959815,86.984619,23.197519
238,Avalon Technologies Limited,24.211131,54.284051,18.445476,18.445476,92.285383,38.667005,159.698387,37.453695


In [17]:
def get_nifty_price (day_offset,time_period):
    """
     For each company's listing_date + day_offset, finds Nifty 50's closing 
    price nearest to that target date. No partitioning needed since Nifty 
    is a single series and we're just finding, for each of the 397 target 
    dates, the closest Nifty price to it.
    """
    query = f"""
    WITH target_dates AS (
    SELECT nse_symbol ,listing_date,
       date(listing_date, '+{day_offset} days') AS target_date
    FROM ipo_clean
    ),
    ranked_nifty AS (
    SELECT 
        t.nse_symbol,
        t.target_date,
        n.Date AS nifty_date,
        n.nifty_close,
        ABS(julianday(n.Date) - julianday(t.target_date)) AS day_diff,
        ROW_NUMBER() OVER (
            PARTITION BY t.nse_symbol
            ORDER BY ABS(julianday(n.Date) - julianday(t.target_date))ASC
        ) AS rank
        FROM target_dates t
        JOIN nifty50 n 
    )
    SELECT
        nse_symbol,
        CASE 
            WHEN day_diff >= 10 THEN NULL
            ELSE nifty_close
        END AS nifty_close_{time_period}
    FROM ranked_nifty
    WHERE rank = 1
    """

    return pd.read_sql(query,conn)            

In [18]:
nifty_at_listing = get_nifty_price(0, 'listing')
print(nifty_at_listing.shape)
nifty_at_listing.head()

(397, 2)


,nse_symbol,nifty_close_listing
0,AAVAS,10348.049805
1,ABDL,24123.849609
2,ABSLAMC,17945.949219
3,ACI,18159.949219
4,ACMESOLAR,23559.050781


In [19]:
nifty_results = {'listing' : nifty_at_listing}

for day_offset,period in [(182, '6mo'), (365, '1yr'), (730, '2yr'), (1095, '3yr')]:
    print(f"Running {period}-")
    nifty_results[period] = get_nifty_price(day_offset,period)
    print(f"Done:{nifty_results[period].shape}")

Running 6mo-
Done:(397, 2)
Running 1yr-
Done:(397, 2)
Running 2yr-
Done:(397, 2)
Running 3yr-
Done:(397, 2)


In [20]:
for period in ['6mo','1yr','2yr','3yr']:
    price_col = f'nifty_close_{period}'
    master = master.merge(nifty_results[period][['nse_symbol',price_col]], on ='nse_symbol', how ='left')

master = master.merge(nifty_results['listing'][['nse_symbol','nifty_close_listing']], on ='nse_symbol', how ='left')

print(master.shape)
master.head()

(397, 23)


,nse_symbol,company_name,listing_date,issue_price,listing_price,pop_pct,close_6mo,close_1yr,close_2yr,close_3yr,...,return_3yr,cagr_6mo,cagr_1yr,cagr_2yr,cagr_3yr,nifty_close_6mo,nifty_close_1yr,nifty_close_2yr,nifty_close_3yr,nifty_close_listing
0,GKSL,Gujarat Kidney & S...,2025-12-30,114,120.75,5.921053,129.330002,NaN,NaN,NaN,...,NaN,14.716077,NaN,NaN,NaN,23946.250000,NaN,NaN,NaN,25938.849609
1,KSHINTL,KSH International ...,2025-12-23,384,370.00,-3.645833,863.400024,NaN,NaN,NaN,...,NaN,444.528563,NaN,NaN,NaN,23824.099609,NaN,NaN,NaN,26177.150391
2,ICICIAMC,ICICI Prudential A...,2025-12-19,2165,2606.20,20.378753,3429.000000,NaN,NaN,NaN,...,NaN,73.108937,NaN,NaN,NaN,24013.099609,NaN,NaN,NaN,25966.400391
3,NEPHROPLUS,Nephrocare Health ...,2025-12-17,460,491.70,6.891304,737.000000,NaN,NaN,NaN,...,NaN,124.664555,NaN,NaN,NaN,24085.699219,NaN,NaN,NaN,25818.550781
4,PARKHOSPS,Park Medi World Lt...,2025-12-17,162,155.60,-3.950617,254.149994,NaN,NaN,NaN,...,NaN,166.784763,NaN,NaN,NaN,24085.699219,NaN,NaN,NaN,25818.550781


In [21]:
print(master.shape)
print(master.columns.tolist())

# Checking an old company that has all Nifty horizons filled
master[master['nifty_close_3yr'].notna()][
    ['company_name', 'listing_date', 'nifty_close_listing', 'nifty_close_6mo', 
     'nifty_close_1yr', 'nifty_close_2yr', 'nifty_close_3yr']
].head(5)

(397, 23)
['nse_symbol', 'company_name', 'listing_date', 'issue_price', 'listing_price', 'pop_pct', 'close_6mo', 'close_1yr', 'close_2yr', 'close_3yr', 'return_6mo', 'return_1yr', 'return_2yr', 'return_3yr', 'cagr_6mo', 'cagr_1yr', 'cagr_2yr', 'cagr_3yr', 'nifty_close_6mo', 'nifty_close_1yr', 'nifty_close_2yr', 'nifty_close_3yr', 'nifty_close_listing']


,company_name,listing_date,nifty_close_listing,nifty_close_6mo,nifty_close_1yr,nifty_close_2yr,nifty_close_3yr
234,Ideaforge Technologies Limited,2023-07-07,19331.800781,21710.800781,24323.849609,25461.300781,23946.250000
235,HMA Agro Industries Limited,2023-07-04,19389.000000,21665.800781,24286.500000,25405.300781,23946.250000
236,IKIO Technologies Limited (formerly IKIO Light...,2023-06-16,18826.000000,21456.650391,23465.599609,24946.500000,23853.900391
237,Mankind Pharma Limited,2023-05-09,18265.949219,19406.699219,22302.500000,24273.800781,24176.150391
238,Avalon Technologies Limited,2023-04-18,17660.150391,19811.500000,21995.849609,23851.650391,24353.550781


In [22]:
#getting Nifty's return over the same period, for the same company's listing date
for period in ['6mo','1yr','2yr','3yr']:
    master[f'nifty_return_{period}'] = (( master[f'nifty_close_{period}']- master['nifty_close_listing']) /  master['nifty_close_listing'])*100

    #company's return relative to nifty
    master[f'relative_return_{period}'] = (master[f'return_{period}'] - master[f'nifty_return_{period}'])

master[['company_name', 'return_1yr', 'nifty_return_1yr', 'relative_return_1yr']].head(100)

,company_name,return_1yr,nifty_return_1yr,relative_return_1yr
0,Gujarat Kidney & S...,NaN,NaN,NaN
1,KSH International ...,NaN,NaN,NaN
2,ICICI Prudential A...,NaN,NaN,NaN
3,Nephrocare Health ...,NaN,NaN,NaN
4,Park Medi World Lt...,NaN,NaN,NaN
...,...,...,...,...
95,Ajax Engineering Limited,-17.959528,12.046867,-30.006395
96,Dr Agarwal's Healthcare Limited,13.466870,8.579673,4.887197
97,Denta Water & Infrastructure Limited,-20.651513,9.738769,-30.390282
98,Stallion India Fluorochem Limited,64.499995,7.943430,56.556565


In [23]:
#getting CAGR(Compound Annual Growth Rate) for nifty over different time periods
years = {'6mo':0.5,'1yr':1,'2yr':2,'3yr':3}

for period,year in years.items():
    master[f'nifty_cagr_{period}'] = (( master[f'nifty_close_{period}']/ master['nifty_close_listing']) ** (1/year) - 1)*100

    master[f'relative_cagr_{period}'] = (master[f'cagr_{period}'] - master[f'nifty_cagr_{period}'])

master[['company_name', 'cagr_1yr', 'nifty_cagr_1yr', 'relative_cagr_1yr']].head(100)

,company_name,cagr_1yr,nifty_cagr_1yr,relative_cagr_1yr
0,Gujarat Kidney & S...,NaN,NaN,NaN
1,KSH International ...,NaN,NaN,NaN
2,ICICI Prudential A...,NaN,NaN,NaN
3,Nephrocare Health ...,NaN,NaN,NaN
4,Park Medi World Lt...,NaN,NaN,NaN
...,...,...,...,...
95,Ajax Engineering Limited,-17.959528,12.046867,-30.006395
96,Dr Agarwal's Healthcare Limited,13.466870,8.579673,4.887197
97,Denta Water & Infrastructure Limited,-20.651513,9.738769,-30.390282
98,Stallion India Fluorochem Limited,64.499995,7.943430,56.556565


In [24]:
print(master.shape)
print(master.columns.tolist())
master.isnull().sum()

(397, 39)
['nse_symbol', 'company_name', 'listing_date', 'issue_price', 'listing_price', 'pop_pct', 'close_6mo', 'close_1yr', 'close_2yr', 'close_3yr', 'return_6mo', 'return_1yr', 'return_2yr', 'return_3yr', 'cagr_6mo', 'cagr_1yr', 'cagr_2yr', 'cagr_3yr', 'nifty_close_6mo', 'nifty_close_1yr', 'nifty_close_2yr', 'nifty_close_3yr', 'nifty_close_listing', 'nifty_return_6mo', 'relative_return_6mo', 'nifty_return_1yr', 'relative_return_1yr', 'nifty_return_2yr', 'relative_return_2yr', 'nifty_return_3yr', 'relative_return_3yr', 'nifty_cagr_6mo', 'relative_cagr_6mo', 'nifty_cagr_1yr', 'relative_cagr_1yr', 'nifty_cagr_2yr', 'relative_cagr_2yr', 'nifty_cagr_3yr', 'relative_cagr_3yr']


nse_symbol               0
company_name             0
listing_date             0
issue_price              0
listing_price            0
pop_pct                  0
close_6mo                0
close_1yr               78
close_2yr              159
close_3yr              234
return_6mo               0
return_1yr              78
return_2yr             159
return_3yr             234
cagr_6mo                 0
cagr_1yr                78
cagr_2yr               159
cagr_3yr               234
nifty_close_6mo          0
nifty_close_1yr         78
nifty_close_2yr        159
nifty_close_3yr        234
nifty_close_listing      0
nifty_return_6mo         0
relative_return_6mo      0
nifty_return_1yr        78
relative_return_1yr     78
nifty_return_2yr       159
relative_return_2yr    159
nifty_return_3yr       234
relative_return_3yr    234
nifty_cagr_6mo           0
relative_cagr_6mo        0
nifty_cagr_1yr          78
relative_cagr_1yr       78
nifty_cagr_2yr         159
relative_cagr_2yr      159
n

In [25]:
master[['return_6mo', 'return_1yr', 'return_2yr', 'return_3yr']].describe()

,return_6mo,return_1yr,return_2yr,return_3yr
count,397.000000,319.000000,238.000000,163.000000
mean,-0.226145,7.825378,17.636689,46.759965
std,55.737891,68.970379,96.434489,135.198786
min,-97.655894,-97.472175,-98.384885,-97.181463
25%,-31.724891,-36.603270,-39.581219,-42.824203
50%,-7.616579,-7.097266,-5.569061,-0.423419
75%,19.884790,35.583903,52.175815,90.900486
max,436.665599,334.587725,649.696762,659.161290


In [26]:
master.to_csv('../data/processed/ipo_master_returns.csv', index = False)
ipo_clean.to_csv('../data/processed/ipo_clean.csv', index = False)
master.to_sql('ipo_master_returns',conn, if_exists = 'replace', index = False)
print(master.shape)

(397, 39)


## Step 3 Summary: SQL Return Calculations

Loaded `ipo_clean`, `price_history`, and `nifty50` into SQLite. Used window 
functions (`ROW_NUMBER() OVER (PARTITION BY ... ORDER BY ...)`) to find, for 
each company and each horizon (6mo/1yr/2yr/3yr), the closing price nearest to 
`listing_date + N days` ,handling weekends/holidays by matching to the 
closest available trading day rather than requiring an exact date match. 
Matches more than 9 days from the target were treated as unavailable (typically 
meaning the horizon hasn't been reached yet relative to the data cutoff).

Computed the same nearest-date lookup against Nifty 50 for each company's 
listing date and each horizon, enabling both raw and market-relative return 
calculations. All returns are measured from **listing-day close**, not issue 
price (per the project's locked methodology, keeping listing-day pop % 
independent of forward returns). Both raw % returns and CAGR were calculated, 
since holding periods vary across companies as CAGR allows fair comparison 
across companies observed for different lengths of time.

**Data quality issues found & resolved:**
- Premier Energies Limited had been matched to the wrong NSE symbol (`PREMIER`, 
  an unrelated company) instead of its correct symbol `PREMIERENE` — caused a 
  false ~-99% return; corrected by re-matching the symbol and re-pulling its 
  price history
- Protean eGov Technologies' `listing_date` reflected its BSE listing date, not 
  its NSE first-trading date, causing its 6-month/1-year horizons to appear 
  unavailable — corrected using the earliest date actually present in its 
  NSE price history

**Output:** `data/processed/ipo_master_returns.csv` — one row per company with 
pop %, raw returns, CAGR, and Nifty-relative performance across all four horizons.